In [193]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

In [194]:
df = pd.read_csv("../data/train.csv")


This is the Shipment attributes Section (Refer to shipment_attr.ipynb) for detailed README
Deals with Duplicate shipment IDs first

In [195]:
n_dup_rows = df.duplicated().sum()
n_dup_ids = df["shipment_id"].duplicated().sum()
print(f"{n_dup_rows} exact duplicate rows, {n_dup_ids} duplicate shipment_ids, "
      f"all exact copies = {n_dup_rows == n_dup_ids}")

dedup = df.drop_duplicates().reset_index(drop=True)
len(dedup)

81 exact duplicate rows, 81 duplicate shipment_ids, all exact copies = True


39949

In [196]:
for col in ["shipment_mode", "carrier", "service_level"]:
    vc = dedup[col].map(repr).value_counts()
    print(f"--- {col}: {len(vc)} raw variants")
    print(vc.to_string(), end="\n\n")

--- shipment_mode: 2 raw variants
shipment_mode
'parcel'         34042
'ltl_freight'     5907

--- carrier: 24 raw variants
carrier
'Falcon Post'            12924
'Northstar Parcel'       10874
'Summit Express'          7638
'Ironclad Freight'        3397
'Vela Freight'            2276
'Cobalt Regional'         1250
'FALCON POST'              215
'NORTHSTAR PARCEL'         210
'falcon post'              169
'  Falcon Post '           156
'  Northstar Parcel '      137
'northstar parcel'         117
'SUMMIT EXPRESS'           115
'  Summit Express '         94
'summit express'            90
'  Ironclad Freight '       59
'IRONCLAD FREIGHT'          53
'ironclad freight'          44
'VELA FREIGHT'              39
'  Vela Freight '           27
'COBALT REGIONAL'           22
'  Cobalt Regional '        16
'cobalt regional'           15
'vela freight'              12

--- service_level: 24 raw variants
service_level
'Ground'               18094
'Expedited'             6570
'TwoDay'        

In [197]:
CARRIERS = ["Falcon Post", "Summit Express", "Northstar Parcel", "Cobalt Regional",
            "Ironclad Freight", "Vela Freight"]
SERVICE_LEVELS = ["Ground", "Expedited", "TwoDay", "Overnight", "LTL Standard", "LTL Guaranteed"]
MODES = ["parcel", "ltl_freight"]

CANONICAL = {
    "carrier": {v.lower(): v for v in CARRIERS},
    "service_level": {v.lower(): v for v in SERVICE_LEVELS},
    "shipment_mode": {v.lower(): v for v in MODES},
}

def canonicalize(s, mapping):
    """Strip whitespace, then match case-insensitively to the canonical spelling."""
    return s.str.strip().str.lower().map(mapping)

rows = []
for col, mapping in CANONICAL.items():
    clean_col = canonicalize(dedup[col], mapping)
    assert clean_col.notna().all(), f"unmapped {col} values"
    noisy = dedup[col] != clean_col
    ws = dedup[col] != dedup[col].str.strip()
    rows.append({"column": col, "raw_variants": dedup[col].nunique(),
                 "clean_values": clean_col.nunique(), "noisy_rows": noisy.sum(),
                 "whitespace_rows": ws.sum(), "case_only_rows": (noisy & ~ws).sum(),
                 "pct_noisy": round(100 * noisy.mean(), 2)})
cat_report = pd.DataFrame(rows).set_index("column")
cat_report

,raw_variants,clean_values,noisy_rows,whitespace_rows,case_only_rows,pct_noisy
column,,,,,,
carrier,24,6,1590,489,1101,3.98
service_level,24,6,1554,492,1062,3.89
shipment_mode,2,2,0,0,0,0.00


In [198]:
for col in ["origin_zip", "dest_zip"]:
    lengths = dedup[col].astype(str).str.len().value_counts().sort_index().to_dict()
    print(f"{col}: dtype={dedup[col].dtype}, digit-length counts={lengths}")

origin_zip: dtype=int64, digit-length counts={4: 1933, 5: 38016}
dest_zip: dtype=int64, digit-length counts={4: 1935, 5: 38014}


In [199]:
for side in ["origin", "dest"]:
    short = dedup[f"{side}_zip"].astype(str).str.len() < 5
    print(side, dedup.loc[short, f"{side}_metro"].value_counts().to_dict())
    print("  sample:", dedup.loc[short, f"{side}_zip"].head(5).tolist())

origin {'Boston MA': 1933}
  sample: [2199, 2121, 2132, 2189, 2131]
dest {'Boston MA': 1935}
  sample: [2182, 2170, 2110, 2114, 2165]


In [200]:
def fix_zip(s):
    return s.astype("Int64").astype(str).str.zfill(5)

# Each metro should map to a single ZIP-3 prefix once fixed
prefix_check = {}
for side in ["origin", "dest"]:
    z = fix_zip(dedup[f"{side}_zip"])
    assert z.str.fullmatch(r"\d{5}").all()
    prefix_check[side] = dedup.assign(zip3=z.str[:3]).groupby(f"{side}_metro")["zip3"].unique()
pd.DataFrame(prefix_check)

,origin,dest
Atlanta GA,[303],[303]
Boston MA,[021],[021]
Charlotte NC,[282],[282]
Chicago IL,[606],[606]
Columbus OH,[432],[432]
Dallas TX,[752],[752]
Denver CO,[802],[802]
Houston TX,[770],[770]
Kansas City MO,[641],[641]
Los Angeles CA,[900],[900]


In [201]:
for side in ["origin", "dest"]:
    dedup[f"{side}_zip"] = fix_zip(dedup[f"{side}_zip"])
for col in ["origin_zip", "dest_zip"]:
    print(col, dedup[col].dtype, dedup[col].str.len().value_counts().to_dict())

origin_zip str {5: 39949}
dest_zip str {5: 39949}


In [202]:
PARCEL_CARRIERS = {"Falcon Post", "Summit Express", "Northstar Parcel", "Cobalt Regional"}
PARCEL_SERVICES = {"Ground", "Expedited", "TwoDay", "Overnight"}
WEST_MOUNTAIN = {"Los Angeles CA", "Phoenix AZ", "Seattle WA", "Denver CO", "Portland OR", "Salt Lake City UT"}
EARTH_RADIUS_MI = 3958.8

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * EARTH_RADIUS_MI * np.arcsin(np.sqrt(a))

def consistency_checks(df):
    """Return {check: n_violations}. Expects canonical carrier/service_level."""
    parcel = df["shipment_mode"] == "parcel"
    dist_calc = haversine_miles(df["origin_lat"], df["origin_lon"], df["dest_lat"], df["dest_lon"])
    zone_ranges = df.groupby("zone")["distance_miles"].agg(["min", "max"]).sort_index()
    cobalt = df["carrier"] == "Cobalt Regional"
    return {
        "carrier vs mode": (parcel != df["carrier"].isin(PARCEL_CARRIERS)).sum(),
        "service vs mode": (parcel != df["service_level"].isin(PARCEL_SERVICES)).sum(),
        "Cobalt off west/mountain lanes": (cobalt & ~(df["origin_metro"].isin(WEST_MOUNTAIN)
                                                      & df["dest_metro"].isin(WEST_MOUNTAIN))).sum(),
        "metro with >1 origin coord": (df.groupby("origin_metro")[["origin_lat", "origin_lon"]].nunique() > 1).any(axis=1).sum(),
        "metro with >1 dest coord": (df.groupby("dest_metro")[["dest_lat", "dest_lon"]].nunique() > 1).any(axis=1).sum(),
        "distance off haversine >1%": (abs(df["distance_miles"] / dist_calc - 1) > 0.01).sum(),
        "lane with >1 distance": (df.groupby(["origin_metro", "dest_metro"])["distance_miles"].nunique() > 1).sum(),
        "overlapping zone ranges": (zone_ranges["min"].iloc[1:].values <= zone_ranges["max"].iloc[:-1].values).sum(),
        "origin == dest metro": (df["origin_metro"] == df["dest_metro"]).sum(),
    }

checked = dedup.copy()
for col, mapping in CANONICAL.items():
    checked[col] = canonicalize(checked[col], mapping)
pd.Series(consistency_checks(checked), name="violations")

carrier vs mode                   0
service vs mode                   0
Cobalt off west/mountain lanes    0
metro with >1 origin coord        0
metro with >1 dest coord          0
distance off haversine >1%        0
lane with >1 distance             0
overlapping zone ranges           0
origin == dest metro              0
Name: violations, dtype: int64

In [203]:
def clean_shipment_attributes(df):
    """Fix shipment-attribute noise: categorical casing/whitespace and lost-leading-zero ZIPs."""
    df = df.copy()
    for col, mapping in CANONICAL.items():
        df[col] = canonicalize(df[col].astype(str), mapping)
        if df[col].isna().any():
            raise ValueError(f"unrecognized {col} values after cleaning")
    for col in ["origin_zip", "dest_zip"]:
        df[col] = fix_zip(df[col])
    return df

clean = clean_shipment_attributes(dedup)

assert set(clean["carrier"]) == set(CARRIERS)
assert set(clean["service_level"]) == set(SERVICE_LEVELS)
assert set(clean["shipment_mode"]) == set(MODES)
for col in ["origin_zip", "dest_zip"]:
    assert clean[col].str.fullmatch(r"\d{5}").all()
assert all(v == 0 for v in consistency_checks(clean).values())
assert clean.equals(clean_shipment_attributes(clean)), "cleaner is not idempotent"
print(f"all checks pass ({len(clean)} rows)")

all checks pass (39949 rows)


Physical and Handeling Section
Unit error conversions fixed, 0 weight rows dropped and missing values filled in with mean

In [204]:
to_impute_ph = ["weight_lbs", "length_in", "width_in", "height_in"]

df = clean[clean["weight_lbs"] != 0].copy()
condition = (df["weight_lbs"] > 150) & (df["pallet_count"] == 0)
df["weight_lbs"] = np.where(condition, df["weight_lbs"] / 453.592, df["weight_lbs"])
for col in to_impute_ph:
    df[col + "_na"] = df[col].isnull()
    df[col] = df[col].fillna(df[col].mean())
    

In [205]:
print(df["shipment_id"].duplicated().sum())                 # should be 0
print(df[to_impute_ph].isnull().sum())                      # should all be 0
print(set(df["carrier"]) == set(CARRIERS))                  # should be True

0
weight_lbs    0
length_in     0
width_in      0
height_in     0
dtype: int64
True


In [206]:
print(df.shape)
print("duplicate column names:", df.columns[df.columns.duplicated()].tolist())
print("duplicate shipment_ids:", df["shipment_id"].duplicated().sum())
print(df.isnull().sum()[df.isnull().sum() > 0])
print(df.dtypes.value_counts())

(39747, 37)
duplicate column names: []
duplicate shipment_ids: 0
Series([], dtype: int64)
float64    14
str        10
int64       9
bool        4
Name: count, dtype: int64


In [207]:
df.to_csv("engineered_features.csv", index=False)